# 12 · Prefill vs decode on the roofline

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 2. Sources: Williams et al., Roofline (CACM 2009); NVIDIA H100 datasheet.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Compute arithmetic intensity and roofline step time for prefill and decode, and find the
batch-size ceiling set by KV reads.

## Theory and equations
$I_{\text{prefill}} \approx 2T/s$; $I_{\text{decode}} \approx \dfrac{2PB}{sP + BTm_{\text{tok}}} \to \dfrac{2P}{Tm_{\text{tok}}}$ as $B\to\infty$.

## Diagram
```
TFLOP/s |            ______________ 989 (compute roof)
        |          /  ^ prefill
        |        /
        |  B=256 * <- KV ceiling
        |   B=1 *     (bandwidth slope 3.35 TB/s)
        +------------------------------ FLOP/byte (ridge = 295)
```

## Implementation

In [ ]:
from inference_lab import kv_math as K
m, g = K.LLAMA3_8B, K.H100_SXM
print("ridge point:", round(g.ridge_bf16), "FLOP/byte")

## Experiment (*theoretical*)

In [ ]:
for B in (1, 8, 32, 128, 256, 1024):
    f, q = K.step_cost(m, B, 1, 2048); t, bound = K.roofline_time(f, q, g)
    print(f"decode B={B:5d}  I={f/q:6.1f}  step={1e3*t:6.2f} ms  {bound:7s}  {B/t:8.0f} tok/s")
for T in (128, 512, 2048, 8192):
    f, q = K.step_cost(m, 1, T, T // 2); t, bound = K.roofline_time(f, q, g)
    print(f"prefill T={T:5d} I={f/q:7.0f}  time={1e3*t:6.2f} ms  {bound}")

## Results: the KV ceiling vs context

In [ ]:
ctx = [512, 2048, 8192, 32768]
B = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
fig, ax = plt.subplots()
for T, a in zip(ctx, [1, .75, .5, .3]):
    I = [K.arithmetic_intensity(*K.step_cost(m, b, 1, T)) for b in B]
    ax.plot(B, I, "o-", color=C["memory"], alpha=a, label=f"T={T}")
ax.axhline(g.ridge_bf16, color=C["ink"], ls="--", lw=1); ax.text(1, g.ridge_bf16 * 1.1, "H100 ridge")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("batch"); ax.set_ylabel("decode FLOP/byte")
ax.legend(); plt.show()

## Interpretation
Decode intensity saturates far below the ridge, and lower at longer contexts. Past the knee,
batching adds latency without adding much throughput.

## Limitations
First-order model: ignores activations, kernel efficiency, and launch overhead.

## Conclusion
Decode is a bandwidth problem; make bytes smaller (quantization, GQA/MLA) or produce more
tokens per byte (speculative decoding).